# 03 — Training / Evaluation Phase Closure v2

Notebook này khép lại pha training/evaluation của nhánh `codex/preprocessing-v4-core`.

Workflow active gồm 5 phần:

1. **Kiểm tra input contract** trên feature table `v2`.
2. **Clean benchmark** để chọn candidate theo `val_auc`, calibration và threshold lock trên `val`.
3. **Model-level nuisance audit** với `AUC_nat` trên real-only `4:4:4 vs 4:2:0`.
4. **Degradation suite** với `AUC_xdeg` trên các phép hậu kỳ bắt buộc.
5. **Family ablation + phase closure summary** để chốt branch nào còn đáng giữ cho vòng tiếp theo.

Artifact được lưu dưới `audit_output/validation/<run_name>/` theo các pha:
- `phase1_clean_benchmark`
- `phase2_model_nuisance`
- `phase3_degradation_suite`
- `phase4_family_ablation`
- `phase5_phase_closure`

Notebook này **không** còn nhúng `feature-audit` dựa trên metadata cũ; nó chỉ orchestration benchmark và audit ở mức model theo active spec.

In [1]:
from __future__ import annotations

from datetime import datetime
import json
import os
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.training import (
    ABLATION_FEATURE_SET_COLUMNS,
    DEGRADATION_SPECS,
    FEATURE_SET_COLUMNS,
    load_training_table,
    run_training_phase_closure,
)

from src.feature_extraction import ALL_FEATURE_KEYS

FEATURE_TABLE = Path(os.getenv('TRAINING_V2_FEATURE_TABLE', PROJECT_ROOT / 'features' / 'feature_extraction_v2_rgb248_exact.csv'))
DEFAULT_RUN_NAME = f"training_v2_phase_closure_{datetime.now().strftime('%Y%m%d')}"
RUN_NAME = os.getenv('TRAINING_V2_RUN_NAME', DEFAULT_RUN_NAME)
OUTPUT_ROOT = PROJECT_ROOT / 'audit_output' / 'validation' / RUN_NAME
PHASE1_DIR = OUTPUT_ROOT / 'phase1_clean_benchmark'
PHASE2_DIR = OUTPUT_ROOT / 'phase2_model_nuisance'
PHASE3_DIR = OUTPUT_ROOT / 'phase3_degradation_suite'
PHASE4_DIR = OUTPUT_ROOT / 'phase4_family_ablation'
PHASE5_DIR = OUTPUT_ROOT / 'phase5_phase_closure'
MODEL_OUTPUT_DIR = PROJECT_ROOT / 'models' / 'param' / RUN_NAME
SUMMARY_PATH = OUTPUT_ROOT / 'summary.json'
FORCE_RERUN = os.getenv('TRAINING_V2_FORCE_RERUN', '0') == '1'
WORKERS = int(os.getenv('TRAINING_V2_WORKERS', '1'))
SHOW_PROGRESS = os.getenv('TRAINING_V2_SHOW_PROGRESS', '1') == '1'

def read_csv_if_exists(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame({'missing_file': [str(path)]})
    return pd.read_csv(path)

def read_json_if_exists(path: Path) -> dict:
    if not path.exists():
        return {'missing_file': str(path)}
    return json.loads(path.read_text(encoding='utf-8'))

print({
    'feature_table': str(FEATURE_TABLE),
    'run_name': RUN_NAME,
    'output_root': str(OUTPUT_ROOT),
    'model_output_dir': str(MODEL_OUTPUT_DIR),
    'force_rerun': FORCE_RERUN,
    'workers': WORKERS,
    'show_progress': SHOW_PROGRESS,
    'clean_feature_sets': list(FEATURE_SET_COLUMNS),
    'ablation_feature_sets': list(ABLATION_FEATURE_SET_COLUMNS),
    'degradation_suite': [spec.name for spec in DEGRADATION_SPECS],
})

{'feature_table': 'C:\\Users\\USER\\Desktop\\ai_detector_img\\features\\feature_extraction_v2_rgb248_exact.csv', 'run_name': 'training_v2_phase_closure_20260403', 'output_root': 'C:\\Users\\USER\\Desktop\\ai_detector_img\\audit_output\\validation\\training_v2_phase_closure_20260403', 'model_output_dir': 'C:\\Users\\USER\\Desktop\\ai_detector_img\\models\\param\\training_v2_phase_closure_20260403', 'force_rerun': False, 'workers': 1, 'show_progress': True, 'clean_feature_sets': ['control_minimal', 'always_on', 'always_on_plus_cfa_raw', 'always_on_plus_cfa_gated', 'full_v2'], 'ablation_feature_sets': ['always_on', 'always_on_plus_cfa_gated', 'always_on_plus_wavelet', 'always_on_plus_ysrm', 'full_v2', 'full_v2_minus_conditional_cfa', 'full_v2_minus_wavelet_decay', 'full_v2_minus_content_adaptive_y_srm', 'full_v2_minus_dark_textured_hetero'], 'degradation_suite': ['jpeg95_420', 'jpeg90_420', 'resize75_bilinear', 'resize50_bilinear', 'resize50_jpeg90_420']}


## 0. Kiểm tra input contract

Cell này chỉ đọc feature table active và xác nhận contract trước khi chạy phase closure.

In [2]:
training_frame = load_training_table(FEATURE_TABLE)
{
    'rows': int(len(training_frame)),
    'feature_version': str(training_frame['feature_version'].iloc[0]),
    'preprocess_version': str(training_frame['preprocess_version'].iloc[0]),
    'split_role_counts': training_frame['split_role'].value_counts().to_dict(),
    'generator_counts': training_frame['generator'].value_counts().to_dict(),
    'feature_column_count': int(len(ALL_FEATURE_KEYS)),
}

{'rows': 85615,
 'feature_version': 'v2_rgb248_exact_multibranch',
 'preprocess_version': 'v4_rgb248_r4_exact',
 'split_role_counts': {'train_core': 44235,
  'ood_eval': 27410,
  'val': 5821,
  'id_test': 5821,
  'calibration': 2328},
 'generator_counts': {'SDv15': 15670,
  'GLIDE': 11740,
  'SDv14': 11729,
  'Wukong': 11727,
  'ADM': 11721,
  'VQDM': 11717,
  'Midjourney': 11311},
 'feature_column_count': 36}

## 0.1. Inventory branch benchmark

Cell này nhắc lại inventory branch nào sẽ được benchmark clean và branch nào sẽ đi vào family ablation.

In [3]:
{
    'clean_feature_sets': {name: len(cols) for name, cols in FEATURE_SET_COLUMNS.items()},
    'ablation_feature_sets': {name: len(cols) for name, cols in ABLATION_FEATURE_SET_COLUMNS.items()},
}

{'clean_feature_sets': {'control_minimal': 8,
  'always_on': 14,
  'always_on_plus_cfa_raw': 19,
  'always_on_plus_cfa_gated': 19,
  'full_v2': 36},
 'ablation_feature_sets': {'always_on': 14,
  'always_on_plus_cfa_gated': 19,
  'always_on_plus_wavelet': 20,
  'always_on_plus_ysrm': 20,
  'full_v2': 36,
  'full_v2_minus_conditional_cfa': 31,
  'full_v2_minus_wavelet_decay': 30,
  'full_v2_minus_content_adaptive_y_srm': 30,
  'full_v2_minus_dark_textured_hetero': 31}}

## 1. Run Or Load Phase Closure

Nếu artifact đã tồn tại và `TRAINING_V2_FORCE_RERUN=0`, notebook sẽ nạp lại kết quả.
Nếu cần chạy lại toàn bộ clean benchmark, nuisance audit, degradation suite và family ablation, đặt `TRAINING_V2_FORCE_RERUN=1`.

In [4]:
if SUMMARY_PATH.exists() and not FORCE_RERUN:
    summary = read_json_if_exists(SUMMARY_PATH)
else:
    summary = run_training_phase_closure(
        FEATURE_TABLE,
        output_root=OUTPUT_ROOT,
        model_output_dir=MODEL_OUTPUT_DIR,
        workers=WORKERS,
        force_rerun=FORCE_RERUN,
        show_progress=SHOW_PROGRESS,
    )
summary

Degrade resize50_jpeg90_420: 100%|██████████| 39052/39052 [33:08<00:00, 19.64img/s]


{'feature_table_path': 'C:\\Users\\USER\\Desktop\\ai_detector_img\\features\\feature_extraction_v2_rgb248_exact.csv',
 'rows': 85615,
 'feature_version': 'v2_rgb248_exact_multibranch',
 'preprocess_version': 'v4_rgb248_r4_exact',
 'selected_clean_candidate': 'full_v2__lightgbm',
 'selected_clean_feature_set': 'full_v2',
 'selected_clean_model_name': 'lightgbm',
 'selected_clean_val_auc': 0.9548272480207964,
 'selected_clean_threshold': 0.7074744498826763,
 'required_audits_completed': True,
 'best_branch_candidate': 'full_v2__lightgbm',
 'best_branch_feature_set': 'full_v2',
 'best_branch_clean_pooled_auc': 0.9628714733532449,
 'best_branch_worst_xdeg_auc': 0.5886909984350508,
 'best_branch_auc_nat_abs': 0.6710345966170204,
 'phase1_files': {'candidate_val_metrics_csv': 'C:\\Users\\USER\\Desktop\\ai_detector_img\\audit_output\\validation\\training_v2_phase_closure_20260403\\phase1_clean_benchmark\\candidate_val_metrics.csv',
  'selected_model_metrics_csv': 'C:\\Users\\USER\\Desktop\\ai

## 2. Clean benchmark trên validation

In [5]:
candidate_metrics = read_csv_if_exists(PHASE1_DIR / 'candidate_val_metrics.csv')
candidate_metrics.sort_values(['val_auc', 'val_brier'], ascending=[False, True]).head(12)

,candidate_name,feature_set,model_name,model_family,feature_count,cfa_threshold,val_auc,val_brier,val_ece,val_threshold,val_tpr,val_fpr,val_precision,val_accuracy
0,full_v2__lightgbm,full_v2,lightgbm,tree,36,-0.532941,0.954827,0.083164,0.014494,0.707474,0.795333,0.049982,0.944203,0.870297
1,always_on_plus_cfa_raw__lightgbm,always_on_plus_cfa_raw,lightgbm,tree,19,-0.532941,0.933964,0.102545,0.012802,0.730323,0.726000,0.049982,0.939198,0.834565
2,full_v2__logreg,full_v2,logreg,linear,36,-0.532941,0.914035,0.116339,0.018238,0.715095,0.675667,0.049982,0.934963,0.808624
3,always_on_plus_cfa_raw__logreg,always_on_plus_cfa_raw,logreg,linear,19,-0.532941,0.904559,0.123456,0.015545,0.716501,0.652667,0.049982,0.932825,0.796770
4,always_on_plus_cfa_gated__lightgbm,always_on_plus_cfa_gated,lightgbm,tree,19,-0.532941,0.868869,0.147525,0.009757,0.776969,0.477000,0.049982,0.910305,0.706236
5,always_on__lightgbm,always_on,lightgbm,tree,14,-0.532941,0.821360,0.172383,0.018942,0.775546,0.384000,0.049982,0.890951,0.658306
6,always_on_plus_cfa_gated__logreg,always_on_plus_cfa_gated,logreg,linear,19,-0.532941,0.777066,0.192458,0.026324,0.739326,0.334667,0.049982,0.876856,0.632881
7,control_minimal__lightgbm,control_minimal,lightgbm,tree,8,-0.532941,0.773032,0.193752,0.017586,0.750622,0.301333,0.049982,0.865072,0.615702
8,always_on__logreg,always_on,logreg,linear,14,-0.532941,0.752388,0.204136,0.048336,0.729890,0.261333,0.049982,0.847568,0.595087
9,control_minimal__logreg,control_minimal,logreg,linear,8,-0.532941,0.708508,0.218617,0.043415,0.714767,0.202667,0.049982,0.811749,0.564851


## 3. Selected model trên clean split

In [6]:
selected_metrics = read_csv_if_exists(PHASE1_DIR / 'selected_model_metrics.csv')
selected_metrics

,split,candidate_name,feature_set,model_name,feature_count,auc_ci_low,auc_ci_high,auc,brier,ece,...,precision,recall,accuracy,tp,tn,fp,fn,n_pos,n_neg,n_total
0,val,full_v2__lightgbm,full_v2,lightgbm,36,0.950817,0.959515,0.954827,0.083164,0.014494,...,0.944203,0.795333,0.870297,2386,2680,141,614,3000,2821,5821
1,id_test,full_v2__lightgbm,full_v2,lightgbm,36,0.944785,0.954154,0.949068,0.089715,0.010927,...,0.940891,0.774667,0.858787,2324,2675,146,676,3000,2821,5821
2,ood_eval,full_v2__lightgbm,full_v2,lightgbm,36,0.965858,0.969356,0.967559,0.070428,0.024139,...,0.946470,0.851701,0.899708,11917,12744,674,2075,13992,13418,27410
3,pooled_eval,full_v2__lightgbm,full_v2,lightgbm,36,0.961181,0.964462,0.962871,0.075201,0.016546,...,0.945360,0.831683,0.889225,16627,18099,961,3365,19992,19060,39052


## 4. OOD breakdown và clean CFA gate coverage

In [7]:
ood_by_generator = read_csv_if_exists(PHASE1_DIR / 'selected_model_ood_by_generator.csv')
clean_cfa_gate_coverage = read_csv_if_exists(PHASE1_DIR / 'clean_cfa_gate_coverage.csv')
ood_by_generator, clean_cfa_gate_coverage

(  generator       auc     brier       ece  threshold       tpr       fpr  \
 0     GLIDE  0.981562  0.055757  0.046376   0.707474  0.915333  0.051394   
 1     SDv15  0.956878  0.081419  0.011266   0.707474  0.803929  0.049362   
 
    precision    recall  accuracy    tp    tn   fp    fn  n_pos  n_neg  n_total  
 0   0.949024  0.915333  0.931601  5492  5445  295   508   6000   5740    11740  
 1   0.944297  0.803929  0.875814  6425  7299  379  1567   7992   7678    15670  ,
     split_role   label  gate_rate  gate_active_count   rows
 0  calibration      ai   0.247500                297   1200
 1  calibration  nature   0.265957                300   1128
 2      id_test      ai   0.231000                693   3000
 3      id_test  nature   0.259128                731   2821
 4     ood_eval      ai   0.087193               1220  13992
 5     ood_eval  nature   0.268743               3606  13418
 6   train_core      ai   0.236184               5385  22800
 7   train_core  nature   0.2647

## 5. Model-level AUC_nat

In [8]:
auc_nat_metrics = read_csv_if_exists(PHASE2_DIR / 'model_level_auc_nat.csv')
auc_nat_metrics.sort_values(['split', 'auc_nat_abs', 'candidate_name'], ascending=[True, False, True])

,candidate_name,feature_set,model_name,split,n_rows,n_420,n_444,auc_nat_raw,auc_nat_abs,pred_mean_420,pred_mean_444,pred_gap_420_minus_444
1,always_on__lightgbm,always_on,lightgbm,id_test,2793,301,2492,0.749013,0.749013,0.552844,0.334929,0.217915
9,always_on_plus_wavelet__lightgbm,always_on_plus_wavelet,lightgbm,id_test,2793,301,2492,0.745894,0.745894,0.535645,0.312488,0.223157
5,always_on_plus_cfa_gated__lightgbm,always_on_plus_cfa_gated,lightgbm,id_test,2793,301,2492,0.725398,0.725398,0.492051,0.280349,0.211702
29,full_v2_minus_content_adaptive_y_srm__lightgbm,full_v2_minus_content_adaptive_y_srm,lightgbm,id_test,2793,301,2492,0.707143,0.707143,0.345398,0.180477,0.164921
17,full_v2__lightgbm,full_v2,lightgbm,id_test,2793,301,2492,0.677106,0.677106,0.289527,0.165429,0.124098
21,full_v2_minus_conditional_cfa__lightgbm,full_v2_minus_conditional_cfa,lightgbm,id_test,2793,301,2492,0.671884,0.671884,0.418285,0.276730,0.141554
33,full_v2_minus_dark_textured_hetero__lightgbm,full_v2_minus_dark_textured_hetero,lightgbm,id_test,2793,301,2492,0.671004,0.671004,0.289768,0.167911,0.121857
25,full_v2_minus_wavelet_decay__lightgbm,full_v2_minus_wavelet_decay,lightgbm,id_test,2793,301,2492,0.668045,0.668045,0.291598,0.169326,0.122271
13,always_on_plus_ysrm__lightgbm,always_on_plus_ysrm,lightgbm,id_test,2793,301,2492,0.645681,0.645681,0.427122,0.304363,0.122760
2,always_on__lightgbm,always_on,lightgbm,ood_eval,13303,1143,12160,0.755304,0.755304,0.554947,0.336697,0.218250


## 5.1. Hỗ trợ nhãn nuisance

In [9]:
nuisance_label_summary = read_csv_if_exists(PHASE2_DIR / 'nuisance_label_summary.csv')
nuisance_label_summary

,split_role,jpeg_subsampling_live,rows
0,calibration,4:2:0,99
1,calibration,4:4:4,1019
2,id_test,4:2:0,301
3,id_test,4:4:4,2492
4,ood_eval,4:2:0,1143
5,ood_eval,4:4:4,12160
6,train_core,4:2:0,2015
7,train_core,4:4:4,19169
8,val,4:2:0,265
9,val,4:4:4,2527


## 6. Degradation suite: metrics gộp

In [10]:
degradation_metrics = read_csv_if_exists(PHASE3_DIR / 'degradation_metrics.csv')
degradation_metrics.loc[degradation_metrics['split'] == 'pooled_eval'].sort_values(['candidate_name', 'auc'], ascending=[True, False])

,split,candidate_name,feature_set,model_name,feature_count,auc_ci_low,auc_ci_high,auc,brier,ece,...,recall,accuracy,tp,tn,fp,fn,n_pos,n_neg,n_total,degradation_name
3,pooled_eval,always_on__lightgbm,always_on,lightgbm,14,0.858231,0.865412,0.861773,0.153535,0.042702,...,0.488145,0.711359,9759,18021,1039,10233,19992,19060,39052,jpeg95_420
39,pooled_eval,always_on__lightgbm,always_on,lightgbm,14,0.838743,0.846288,0.842285,0.163685,0.039490,...,0.436625,0.685829,8729,18054,1006,11263,19992,19060,39052,jpeg90_420
75,pooled_eval,always_on__lightgbm,always_on,lightgbm,14,0.768086,0.777490,0.773046,0.341905,0.355677,...,0.900810,0.620250,18009,6213,12847,1983,19992,19060,39052,resize75_bilinear
147,pooled_eval,always_on__lightgbm,always_on,lightgbm,14,0.628408,0.639611,0.633855,0.423530,0.424289,...,0.975640,0.515748,19505,636,18424,487,19992,19060,39052,resize50_jpeg90_420
111,pooled_eval,always_on__lightgbm,always_on,lightgbm,14,0.618662,0.629236,0.624077,0.455169,0.456654,...,0.991597,0.513981,19824,248,18812,168,19992,19060,39052,resize50_bilinear
7,pooled_eval,always_on_plus_cfa_gated__lightgbm,always_on_plus_cfa_gated,lightgbm,19,0.846436,0.853902,0.849797,0.180357,0.128402,...,0.373649,0.662578,7470,18405,655,12522,19992,19060,39052,jpeg95_420
43,pooled_eval,always_on_plus_cfa_gated__lightgbm,always_on_plus_cfa_gated,lightgbm,19,0.832823,0.840337,0.836507,0.177318,0.098140,...,0.350090,0.648725,6999,18335,725,12993,19992,19060,39052,jpeg90_420
79,pooled_eval,always_on_plus_cfa_gated__lightgbm,always_on_plus_cfa_gated,lightgbm,19,0.737957,0.748133,0.743108,0.306778,0.296941,...,0.829282,0.643527,16579,8552,10508,3413,19992,19060,39052,resize75_bilinear
115,pooled_eval,always_on_plus_cfa_gated__lightgbm,always_on_plus_cfa_gated,lightgbm,19,0.654288,0.664358,0.659632,0.410284,0.412720,...,0.961285,0.534672,19218,1662,17398,774,19992,19060,39052,resize50_bilinear
151,pooled_eval,always_on_plus_cfa_gated__lightgbm,always_on_plus_cfa_gated,lightgbm,19,0.643859,0.654841,0.649657,0.391532,0.392252,...,0.943577,0.549140,18864,2581,16479,1128,19992,19060,39052,resize50_jpeg90_420


## 6.1. Degradation gaps so với clean

In [11]:
degradation_gap_summary = read_csv_if_exists(PHASE3_DIR / 'degradation_gap_summary.csv')
degradation_gap_summary.loc[degradation_gap_summary['split'] == 'pooled_eval'].sort_values(['candidate_name', 'auc_gap'])

,candidate_name,feature_set,model_name,degradation_name,split,clean_auc,auc,auc_gap,clean_brier,brier,brier_gap,clean_ece,ece,ece_gap,clean_tpr,tpr,tpr_gap,clean_fpr,fpr,fpr_gap
10,always_on__lightgbm,always_on,lightgbm,resize50_bilinear,pooled_eval,0.878468,0.624077,-0.254391,0.145581,0.455169,0.309588,0.054645,0.456654,0.402009,0.556273,0.991597,0.435324,0.055299,0.986988,0.931689
14,always_on__lightgbm,always_on,lightgbm,resize50_jpeg90_420,pooled_eval,0.878468,0.633855,-0.244613,0.145581,0.423530,0.277948,0.054645,0.424289,0.369644,0.556273,0.975640,0.419368,0.055299,0.966632,0.911333
18,always_on__lightgbm,always_on,lightgbm,resize75_bilinear,pooled_eval,0.878468,0.773046,-0.105422,0.145581,0.341905,0.196324,0.054645,0.355677,0.301031,0.556273,0.900810,0.344538,0.055299,0.674029,0.618730
2,always_on__lightgbm,always_on,lightgbm,jpeg90_420,pooled_eval,0.878468,0.842285,-0.036183,0.145581,0.163685,0.018104,0.054645,0.039490,-0.015156,0.556273,0.436625,-0.119648,0.055299,0.052781,-0.002518
6,always_on__lightgbm,always_on,lightgbm,jpeg95_420,pooled_eval,0.878468,0.861773,-0.016695,0.145581,0.153535,0.007953,0.054645,0.042702,-0.011943,0.556273,0.488145,-0.068127,0.055299,0.054512,-0.000787
34,always_on_plus_cfa_gated__lightgbm,always_on_plus_cfa_gated,lightgbm,resize50_jpeg90_420,pooled_eval,0.899724,0.649657,-0.250067,0.130150,0.391532,0.261381,0.034255,0.392252,0.357996,0.573930,0.943577,0.369648,0.052466,0.864586,0.812120
30,always_on_plus_cfa_gated__lightgbm,always_on_plus_cfa_gated,lightgbm,resize50_bilinear,pooled_eval,0.899724,0.659632,-0.240092,0.130150,0.410284,0.280134,0.034255,0.412720,0.378464,0.573930,0.961285,0.387355,0.052466,0.912802,0.860336
38,always_on_plus_cfa_gated__lightgbm,always_on_plus_cfa_gated,lightgbm,resize75_bilinear,pooled_eval,0.899724,0.743108,-0.156617,0.130150,0.306778,0.176627,0.034255,0.296941,0.262686,0.573930,0.829282,0.255352,0.052466,0.551312,0.498846
22,always_on_plus_cfa_gated__lightgbm,always_on_plus_cfa_gated,lightgbm,jpeg90_420,pooled_eval,0.899724,0.836507,-0.063217,0.130150,0.177318,0.047168,0.034255,0.098140,0.063885,0.573930,0.350090,-0.223840,0.052466,0.038038,-0.014428
26,always_on_plus_cfa_gated__lightgbm,always_on_plus_cfa_gated,lightgbm,jpeg95_420,pooled_eval,0.899724,0.849797,-0.049927,0.130150,0.180357,0.050206,0.034255,0.128402,0.094147,0.573930,0.373649,-0.200280,0.052466,0.034365,-0.018101


## 6.2. CFA gate coverage dưới degradation

In [12]:
degradation_cfa_gate_coverage = read_csv_if_exists(PHASE3_DIR / 'degradation_cfa_gate_coverage.csv')
degradation_cfa_gate_coverage.sort_values(['degradation_name', 'split_role', 'label'])

,split_role,label,gate_rate,gate_active_count,rows,degradation_name
6,id_test,ai,0.006667,20,3000,jpeg90_420
7,id_test,nature,0.003545,10,2821,jpeg90_420
8,ood_eval,ai,0.004002,56,13992,jpeg90_420
9,ood_eval,nature,0.004173,56,13418,jpeg90_420
10,val,ai,0.009333,28,3000,jpeg90_420
11,val,nature,0.003899,11,2821,jpeg90_420
0,id_test,ai,0.002333,7,3000,jpeg95_420
1,id_test,nature,0.003545,10,2821,jpeg95_420
2,ood_eval,ai,0.001286,18,13992,jpeg95_420
3,ood_eval,nature,0.003428,46,13418,jpeg95_420


## 7. Family ablation trên clean split

In [13]:
ablation_candidate_metrics = read_csv_if_exists(PHASE4_DIR / 'ablation_candidate_val_metrics.csv')
ablation_candidate_metrics.sort_values(['val_auc', 'val_brier'], ascending=[False, True])

,candidate_name,feature_set,model_name,model_family,feature_count,cfa_threshold,val_auc,val_brier,val_ece,val_threshold,val_tpr,val_fpr,val_precision,val_accuracy
0,full_v2__lightgbm,full_v2,lightgbm,tree,36,-0.532941,0.954827,0.083164,0.014494,0.707474,0.795333,0.049982,0.944203,0.870297
1,full_v2_minus_wavelet_decay__lightgbm,full_v2_minus_wavelet_decay,lightgbm,tree,30,-0.532941,0.951224,0.086916,0.008213,0.725209,0.779333,0.049982,0.943122,0.862051
2,full_v2_minus_dark_textured_hetero__lightgbm,full_v2_minus_dark_textured_hetero,lightgbm,tree,31,-0.532941,0.951216,0.086944,0.012544,0.704836,0.785333,0.049982,0.943532,0.865143
3,full_v2_minus_content_adaptive_y_srm__lightgbm,full_v2_minus_content_adaptive_y_srm,lightgbm,tree,30,-0.532941,0.944141,0.094242,0.012914,0.728654,0.753333,0.049982,0.941274,0.848651
4,full_v2_minus_conditional_cfa__lightgbm,full_v2_minus_conditional_cfa,lightgbm,tree,31,-0.532941,0.889689,0.134749,0.012801,0.792812,0.531333,0.049982,0.918732,0.734238
5,always_on_plus_cfa_gated__lightgbm,always_on_plus_cfa_gated,lightgbm,tree,19,-0.532941,0.868869,0.147525,0.009757,0.776969,0.477000,0.049982,0.910305,0.706236
6,always_on_plus_ysrm__lightgbm,always_on_plus_ysrm,lightgbm,tree,20,-0.532941,0.862875,0.150849,0.016047,0.805556,0.475000,0.049982,0.909962,0.705205
7,always_on_plus_wavelet__lightgbm,always_on_plus_wavelet,lightgbm,tree,20,-0.532941,0.842226,0.161992,0.015348,0.778331,0.414333,0.049982,0.898121,0.673939
8,always_on__lightgbm,always_on,lightgbm,tree,14,-0.532941,0.821360,0.172383,0.018942,0.775546,0.384000,0.049982,0.890951,0.658306


## 7.1. Clean pooled metrics của các branch ablation

In [14]:
ablation_clean_metrics = read_csv_if_exists(PHASE4_DIR / 'ablation_clean_metrics.csv')
ablation_clean_metrics.loc[ablation_clean_metrics['split'] == 'pooled_eval'].sort_values(['auc', 'brier'], ascending=[False, True])

,split,candidate_name,feature_set,model_name,feature_count,auc_ci_low,auc_ci_high,auc,brier,ece,...,precision,recall,accuracy,tp,tn,fp,fn,n_pos,n_neg,n_total
19,pooled_eval,full_v2__lightgbm,full_v2,lightgbm,36,0.961181,0.964462,0.962871,0.075201,0.016546,...,0.945360,0.831683,0.889225,16627,18099,961,3365,19992,19060,39052
27,pooled_eval,full_v2_minus_wavelet_decay__lightgbm,full_v2_minus_wavelet_decay,lightgbm,30,0.958274,0.961735,0.959992,0.078238,0.016446,...,0.944871,0.819578,0.883156,16385,18104,956,3607,19992,19060,39052
35,pooled_eval,full_v2_minus_dark_textured_hetero__lightgbm,full_v2_minus_dark_textured_hetero,lightgbm,31,0.958138,0.961517,0.959833,0.078191,0.014716,...,0.944193,0.825130,0.885512,16496,18085,975,3496,19992,19060,39052
31,pooled_eval,full_v2_minus_content_adaptive_y_srm__lightgbm,full_v2_minus_content_adaptive_y_srm,lightgbm,30,0.948369,0.952076,0.950171,0.088149,0.014975,...,0.941538,0.766106,0.855910,15316,18109,951,4676,19992,19060,39052
23,pooled_eval,full_v2_minus_conditional_cfa__lightgbm,full_v2_minus_conditional_cfa,lightgbm,31,0.925705,0.930192,0.927908,0.112162,0.051876,...,0.936043,0.688876,0.816629,13772,18119,941,6220,19992,19060,39052
15,pooled_eval,always_on_plus_ysrm__lightgbm,always_on_plus_ysrm,lightgbm,20,0.905075,0.910598,0.907612,0.126413,0.052094,...,0.932799,0.628351,0.786567,12562,18155,905,7430,19992,19060,39052
7,pooled_eval,always_on_plus_cfa_gated__lightgbm,always_on_plus_cfa_gated,lightgbm,19,0.897057,0.902820,0.899724,0.130150,0.034255,...,0.919833,0.573930,0.756274,11474,18060,1000,8518,19992,19060,39052
11,pooled_eval,always_on_plus_wavelet__lightgbm,always_on_plus_wavelet,lightgbm,20,0.888491,0.894307,0.891401,0.137094,0.049472,...,0.919197,0.586084,0.761728,11717,18030,1030,8275,19992,19060,39052
3,pooled_eval,always_on__lightgbm,always_on,lightgbm,14,0.875125,0.881707,0.878468,0.145581,0.054645,...,0.913429,0.556273,0.745852,11121,18006,1054,8871,19992,19060,39052


## 8. Branch closure summary

In [15]:
branch_closure_summary = read_csv_if_exists(PHASE4_DIR / 'branch_closure_summary.csv')
branch_closure_summary.sort_values(['clean_pooled_auc', 'mean_xdeg_auc', 'auc_nat_abs'], ascending=[False, False, True])

,candidate_name,feature_set,model_name,val_auc,val_brier,val_ece,clean_pooled_auc,clean_pooled_brier,clean_pooled_ece,clean_pooled_tpr,...,auc_nat_raw,auc_nat_abs,pred_gap_420_minus_444,worst_xdeg_auc,mean_xdeg_auc,worst_auc_gap,mean_auc_gap,uses_research_wavelet,uses_research_ysrm,uses_conditional_cfa
0,full_v2__lightgbm,full_v2,lightgbm,0.954827,0.083164,0.014494,0.962871,0.075201,0.016546,0.831683,...,0.671035,0.671035,0.128401,0.588691,0.728091,-0.374180,-0.234781,True,True,True
1,full_v2_minus_wavelet_decay__lightgbm,full_v2_minus_wavelet_decay,lightgbm,0.951224,0.086916,0.008213,0.959992,0.078238,0.016446,0.819578,...,0.664802,0.664802,0.126117,0.597687,0.727981,-0.362305,-0.232011,False,True,True
2,full_v2_minus_dark_textured_hetero__lightgbm,full_v2_minus_dark_textured_hetero,lightgbm,0.951216,0.086944,0.012544,0.959833,0.078191,0.014716,0.825130,...,0.663772,0.663772,0.124146,0.579928,0.720186,-0.379905,-0.239648,True,True,True
3,full_v2_minus_content_adaptive_y_srm__lightgbm,full_v2_minus_content_adaptive_y_srm,lightgbm,0.944141,0.094242,0.012914,0.950171,0.088149,0.014975,0.766106,...,0.702737,0.702737,0.163141,0.613867,0.740164,-0.336305,-0.210007,True,False,True
4,full_v2_minus_conditional_cfa__lightgbm,full_v2_minus_conditional_cfa,lightgbm,0.889689,0.134749,0.012801,0.927908,0.112162,0.051876,0.688876,...,0.668082,0.668082,0.146842,0.586208,0.738780,-0.341700,-0.189129,True,True,False
6,always_on_plus_ysrm__lightgbm,always_on_plus_ysrm,lightgbm,0.862875,0.150849,0.016047,0.907612,0.126413,0.052094,0.628351,...,0.643436,0.643436,0.123360,0.587858,0.733738,-0.319754,-0.173873,False,True,False
5,always_on_plus_cfa_gated__lightgbm,always_on_plus_cfa_gated,lightgbm,0.868869,0.147525,0.009757,0.899724,0.130150,0.034255,0.573930,...,0.720761,0.720761,0.207782,0.649657,0.747740,-0.250067,-0.151984,False,False,True
7,always_on_plus_wavelet__lightgbm,always_on_plus_wavelet,lightgbm,0.842226,0.161992,0.015348,0.891401,0.137094,0.049472,0.586084,...,0.752964,0.752964,0.227082,0.599585,0.737860,-0.291816,-0.153541,True,False,False
8,always_on__lightgbm,always_on,lightgbm,0.821360,0.172383,0.018942,0.878468,0.145581,0.054645,0.556273,...,0.756451,0.756451,0.219526,0.624077,0.747007,-0.254391,-0.131461,False,False,False


## 9. Phase closure manifest

In [16]:
phase_closure_manifest = read_json_if_exists(PHASE5_DIR / 'phase_closure_summary.json')
phase_closure_manifest

{'feature_table_path': 'C:\\Users\\USER\\Desktop\\ai_detector_img\\features\\feature_extraction_v2_rgb248_exact.csv',
 'rows': 85615,
 'feature_version': 'v2_rgb248_exact_multibranch',
 'preprocess_version': 'v4_rgb248_r4_exact',
 'selected_clean_candidate': 'full_v2__lightgbm',
 'selected_clean_feature_set': 'full_v2',
 'selected_clean_model_name': 'lightgbm',
 'selected_clean_val_auc': 0.9548272480207964,
 'selected_clean_threshold': 0.7074744498826763,
 'required_audits_completed': True,
 'best_branch_candidate': 'full_v2__lightgbm',
 'best_branch_feature_set': 'full_v2',
 'best_branch_clean_pooled_auc': 0.9628714733532449,
 'best_branch_worst_xdeg_auc': 0.5886909984350508,
 'best_branch_auc_nat_abs': 0.6710345966170204,
 'phase1_files': {'candidate_val_metrics_csv': 'C:\\Users\\USER\\Desktop\\ai_detector_img\\audit_output\\validation\\training_v2_phase_closure_20260403\\phase1_clean_benchmark\\candidate_val_metrics.csv',
  'selected_model_metrics_csv': 'C:\\Users\\USER\\Desktop\\ai